# 位相キックバック

ユニタリゲート $U$ とその固有状態 $\lvert\psi\rangle$ を考えます。
ユニタリゲートの性質から、固有値はある位相角 $\theta$ を用いて $e^{i\theta}$ と表せます。

$$
U\lvert\psi\rangle = e^{i \theta} \lvert\psi\rangle
$$

この $\theta$ を求めたいとします。これは全体位相として現れるため、直接測定するのは困難です。

位相キックバックは、$\theta$ の情報を $\lvert\psi\rangle$ とは別の量子ビットに取り出すテクニックです。
そうすれば、その量子ビットを測定することで $\theta$ を知ることができます。
このために、$U$ の制御ゲートである $CU$ を使用します。

簡単な例を考えてみましょう。

まず、以下のようにユニタリゲートを設定します。

$$
U(\theta) = \begin{pmatrix}
1 & 0 \\
0 & e^{i\theta} \\
\end{pmatrix}
$$

この場合、$\lvert 1\rangle$ が固有状態です。固有値は $e^{i\theta}$ です。

$$
U(\theta)\lvert 1 \rangle = e^{i\theta}\lvert 1 \rangle
$$

制御ユニタリゲートは
$$
CU(\theta) = CR(\theta) = \begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & e^{i\theta} \\
\end{pmatrix}
$$

位相キックバックは以下の量子回路を使用します。

<img src="img/120_img.png" width="40%">

まず、補助量子ビットにHゲートを作用させて重ね合わせ状態を作ります。

$$
\lvert\psi_1\rangle = \frac{1}{\sqrt{2}}(\lvert0\rangle + \lvert1\rangle) \otimes \lvert1\rangle
$$

次に、補助量子ビットを制御量子ビットとして固有状態に $U(\theta)$ を作用させます。
これにより、固有値を係数として取り出すことができます。

$$
\begin{align}
\lvert\psi_2\rangle &= \frac{1}{\sqrt{2}}(\lvert0\rangle \otimes \lvert1\rangle + \lvert1\rangle \otimes U(\theta)\lvert1\rangle) \\
&= \frac{1}{\sqrt{2}}(\lvert0\rangle \otimes \lvert1\rangle + \lvert1\rangle \otimes e^{i\theta}\lvert1\rangle) \\
&= \frac{1}{\sqrt{2}}(\lvert0\rangle \otimes \lvert1\rangle + e^{i\theta}\lvert1\rangle \otimes \lvert1\rangle) \\
&= \frac{1}{\sqrt{2}}(\lvert0\rangle + e^{i\theta} \lvert1\rangle) \otimes \lvert1\rangle)
\end{align}
$$

以上のように、補助量子ビットの重ね合わせ状態の係数として $\theta$ を取り出すことができました。

ここでは補助量子ビットのみに着目します。
補助量子ビットをそのまま測定すると $\lvert0\rangle$ と $\lvert1\rangle$ が等確率で観測されてしまうため、もう一度 $H$ ゲートを作用させます。

$$
\begin{align}
H\cdot \frac{1}{\sqrt{2}}(\lvert0\rangle + e^{i\theta} \lvert1\rangle) &= \frac{1}{\sqrt{2}} 
\begin{pmatrix} 1 & 1
\\ 1 & -1 \\
\end{pmatrix}
\cdot
\frac{1}{\sqrt{2}}
\begin{pmatrix} 
1 \\ e^{i\theta} \\ 
\end{pmatrix} \\
&= \frac{1}{2} 
\begin{pmatrix} 
1 + e^{i\theta} \\ 1 - e^{i\theta} \\ 
\end{pmatrix}
\end{align}
$$

$H$ ゲートを使うことで、相対位相の差を振幅の差に変換することができました。
最後に、補助量子ビットを繰り返し測定したとき、$\lvert0\rangle$ が観測される確率 $\rm{Pr}(\lvert0\rangle)$ は

$$
\begin{align}
\mathrm{Pr}(\lvert0\rangle) &= \biggl| \frac{1+e^{i\theta}}{2} \biggr|^2 \\
&= \frac{1}{2}(1 + \cos\theta)
\end{align}
$$

となります。これより、固有値の実部を推定することができます。

Blueqatで上記を実装してみましょう。

In [ ]:
from blueqat import Circuit
import numpy as np

上記の量子回路を実装します。
$\theta$ は乱数によって決定されます。

In [ ]:
theta = np.random.rand() * np.pi
shots = 1024

c = Circuit(2)

c.x[1].h[0].cr(theta)[0, 1].h[0].m[0]
res = c.run(shots = shots)

実行結果から、補助量子ビットで測定された $\lvert 0\rangle$ の割合を求めます。
そこから計算した推定値 $\theta_{est}$ を、あらかじめ乱数で決めた $\theta$ と比較します。

In [ ]:
p0 = res['00'] / shots
theta_est = np.arccos(2 * p0 - 1)

In [ ]:
print("estimated θ　 :", theta_est)
print("actual θ :", theta)

以上により、位相キックバックによって $\theta$ を推定することができました。
推定の精度はショット数に依存します。